# 04 — Bina Yüksekliği: Imputation + Grid + GHSL Doğrulama

İmar verisinde **5,787 / 18,575** kayıtta `kat_adedi` eksik (%31). Bu notebook:

1. **Kat → metre** dönüşümü (TS 9111: konut 3.0 m).
2. **Buffer cascade imputation** (10 → 100 → 500 → 1000 m, min 3 komşu) ile eksikleri doldurur.
3. **5-fold CV** ile imputation kalitesini ölçer (RMSE, MAE, R²).
4. **GHSL** (JRC global building height, 2018) ile cross-validation yapar.
5. **30 m grid'e** bina sayısı, ortalama yükseklik, yapı yoğunluğu kolonlarını ekler.

**Önkoşullar:**
- `data/processed/imar_pilot_known.gpkg` + `imar_pilot_unknown.gpkg` (00_data_overview)
- `data/processed/grid_30m_variables.gpkg` (03_variables)
- GEE auth (GHSL için)

**Çıktılar:**
- `data/processed/buildings_pilot_imputed.gpkg` — tüm 18,575 nokta + height_m
- `data/processed/ghsl_built_h_2018.tif` — JRC GHSL raster
- `data/processed/grid_30m_building.gpkg` — 30m grid + 3 yeni kolon

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_PROCESSED, DATA_GRID, FIGURES, TABLES,
    CRS_PROJECTED, GEE_PROJECT,
    FLOOR_HEIGHT_M, BUFFER_CASCADE_M, MIN_NEIGHBORS_FOR_IMPUTATION,
    GRID_30M_VARIABLES, BUILDINGS_IMPUTED,
    GHSL_BUILT_H_RASTER, GHSL_YEAR, GRID_30M_BUILDING,
)
from src.gee_utils import init_ee, boundary_to_ee_geometry, ghsl_built_height
from src.building_height import (
    floors_to_height, buffer_cascade_impute,
    cv_imputation, building_metrics_per_cell,
)
from src.variables import zonal_stats_to_grid

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Pilot known + unknown noktaları yükle ---
known = gpd.read_file(DATA_PROCESSED / "imar_pilot_known.gpkg")
unknown = gpd.read_file(DATA_PROCESSED / "imar_pilot_unknown.gpkg")
print(f"known   : {len(known):,} (kat_numeric mevcut)")
print(f"unknown : {len(unknown):,} (kat_adedi = '-')")
print(f"toplam  : {len(known) + len(unknown):,}")
print(f"\nCRS: {known.crs}")
print(f"\nknown.kat_numeric özeti:\n{known['kat_numeric'].describe()}")

In [ ]:
# --- Kat → metre dönüşümü (default 3.0 m, TS 9111 konut) ---
default_floor_h = FLOOR_HEIGHT_M["default"]
known["height_m"] = floors_to_height(known["kat_numeric"], floor_height_m=default_floor_h)

print(f"Kat → metre (× {default_floor_h} m):")
print(known["height_m"].describe().round(2))

# Aşırı uçları kontrol et — 50+ kat var mı?
extreme = known[known["kat_numeric"] > 30]
if len(extreme) > 0:
    print(f"\n⚠ {len(extreme)} kayıtta kat_numeric > 30 (gökdelen-gibi). Veri girişi hatası olabilir.")

In [ ]:
# --- 5-fold CV: imputation kalitesi ---
print(f"Cross-validation (5-fold, buffers={BUFFER_CASCADE_M}, min_neighbors={MIN_NEIGHBORS_FOR_IMPUTATION})...")

cv = cv_imputation(
    known=known,
    height_col="height_m",
    buffers_m=BUFFER_CASCADE_M,
    min_neighbors=MIN_NEIGHBORS_FOR_IMPUTATION,
    n_splits=5,
    random_state=42,
)

print(f"\nCV sonuçları (n={cv['n_total']:,}):")
print(f"  RMSE: {cv['rmse']:.2f} m")
print(f"  MAE:  {cv['mae']:.2f} m")
print(f"  R²:   {cv['r2']:.3f}")
print(f"\nKullanılan buffer dağılımı:")
for buf, count in cv["buffer_distribution"].items():
    label = f"{buf} m" if buf > 0 else "global fallback"
    pct = 100 * count / cv['n_total']
    print(f"  {label:>15s}: {count:>6,} (%{pct:.1f})")

In [ ]:
# --- CV görsel: gerçek vs tahmin ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(cv["actuals"], cv["predicted"], s=2, alpha=0.3, color="#264653")
lim = max(cv["actuals"].max(), cv["predicted"].max())
axes[0].plot([0, lim], [0, lim], "r--", linewidth=1, label="y = x")
axes[0].set_xlabel("Gerçek yükseklik (m)")
axes[0].set_ylabel("Tahmin (m)")
axes[0].set_title(f"CV: actual vs predicted (R²={cv['r2']:.3f}, RMSE={cv['rmse']:.2f}m)")
axes[0].legend()

residuals = cv["predicted"] - cv["actuals"]
axes[1].hist(residuals, bins=60, color="#E76F51", edgecolor="black", alpha=0.85)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Hata (predicted - actual, m)")
axes[1].set_ylabel("Sayı")
axes[1].set_title(f"Hata dağılımı (medyan={np.median(residuals):.2f}m)")

plt.tight_layout()
plt.savefig(FIGURES / "04_imputation_cv.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Asıl imputation: unknown noktalara değer ata ---
print("Imputation: unknown noktalara buffer cascade ile değer ata...")
imputed_h, used_buf = buffer_cascade_impute(
    known=known,
    unknown=unknown,
    height_col="height_m",
    buffers_m=BUFFER_CASCADE_M,
    min_neighbors=MIN_NEIGHBORS_FOR_IMPUTATION,
)

unknown = unknown.copy()
unknown["height_m"] = imputed_h
unknown["impute_buffer_m"] = used_buf
unknown["impute_source"] = "imputed"

known = known.copy()
known["impute_buffer_m"] = 0
known["impute_source"] = "measured"

print(f"\nUnknown imputed buffer dağılımı:")
for buf, count in pd.Series(used_buf).value_counts().sort_index().items():
    label = f"{buf} m" if buf > 0 else "global fallback"
    pct = 100 * count / len(unknown)
    print(f"  {label:>15s}: {count:>6,} (%{pct:.1f})")

print(f"\nImputed yükseklik özeti:\n{unknown['height_m'].describe().round(2)}")

In [ ]:
# --- known + unknown'ı birleştir ve kaydet ---
common_cols = [c for c in ["mahalle", "sokak", "kapi_no", "parsel_id",
                            "kat_numeric", "height_m",
                            "impute_buffer_m", "impute_source", "geometry"]
               if c in known.columns and c in unknown.columns]
buildings = pd.concat([known[common_cols], unknown[common_cols]], ignore_index=True)
buildings = gpd.GeoDataFrame(buildings, geometry="geometry", crs=known.crs)

print(f"Birleşik bina noktası: {len(buildings):,}")
print(f"  measured: {(buildings['impute_source']=='measured').sum():,}")
print(f"  imputed:  {(buildings['impute_source']=='imputed').sum():,}")

buildings.to_file(BUILDINGS_IMPUTED, driver="GPKG")
print(f"\nKaydedildi: {BUILDINGS_IMPUTED} ({BUILDINGS_IMPUTED.stat().st_size/1024/1024:.2f} MB)")

In [ ]:
# --- 30m grid'e bina metrikleri ekle ---
grid = gpd.read_file(GRID_30M_VARIABLES)
print(f"Grid yüklendi: {len(grid):,} hücre")

grid = building_metrics_per_cell(
    buildings=buildings,
    grid=grid,
    height_col="height_m",
    cell_id_col="cell_id",
)

print(f"\nbuilding_count: {grid['building_count'].describe().round(2).to_dict()}")
print(f"\nHücre kapsama:")
print(f"  Bina olan hücre: {(grid['building_count']>0).sum():,} / {len(grid):,}")
print(f"  Bina yoğunluğu medyan: {grid['building_density_per_km2'].median():.1f} bina/km²")
print(f"  Ortalama yükseklik (bina olan hücrelerde) medyan: {grid['building_height_mean'].median():.2f} m")

In [ ]:
# --- GHSL building height (GEE) ---
import ee
project_id = GEE_PROJECT or os.environ.get("GEE_PROJECT")
if not project_id:
    raise RuntimeError("GEE_PROJECT yok — Notebook 02'deki talimatları izle.")
init_ee(project_id)

boundary = gpd.read_file(DATA_GRID / "pilot_boundary.gpkg")
region_ee = boundary_to_ee_geometry(boundary)

if GHSL_BUILT_H_RASTER.exists():
    print(f"Atlandı (mevcut): {GHSL_BUILT_H_RASTER.name}")
else:
    import geemap
    img = ghsl_built_height(region_ee, year=GHSL_YEAR)
    print(f"İndiriliyor → {GHSL_BUILT_H_RASTER.name}")
    geemap.ee_export_image(
        ee_object=img,
        filename=str(GHSL_BUILT_H_RASTER),
        scale=100,
        region=region_ee,
        file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"  {GHSL_BUILT_H_RASTER.stat().st_size/1024:.1f} KB")

In [ ]:
# --- GHSL'i 30m grid'e zonal mean olarak ekle (karşılaştırma için) ---
import rasterio
with rasterio.open(GHSL_BUILT_H_RASTER) as src:
    ghsl_nodata = src.nodata
print(f"GHSL raster nodata: {ghsl_nodata}")

grid = zonal_stats_to_grid(
    grid=grid,
    raster_path=GHSL_BUILT_H_RASTER,
    stats=("mean",),
    prefix="ghsl_",
    nodata=ghsl_nodata,
)

# İmar-kaynaklı vs GHSL — sadece bina olan hücrelerde karşılaştır
mask = (grid["building_count"] > 0) & grid["ghsl_mean"].notna() & grid["building_height_mean"].notna()
comp = grid.loc[mask, ["building_height_mean", "ghsl_mean"]]

from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = float(np.sqrt(mean_squared_error(comp["ghsl_mean"], comp["building_height_mean"])))
mae = float(mean_absolute_error(comp["ghsl_mean"], comp["building_height_mean"]))
r = float(comp.corr().iloc[0, 1])

print(f"\nİmar-kaynaklı vs GHSL (n={len(comp):,} hücre, bina olan):")
print(f"  Pearson r: {r:.3f}")
print(f"  RMSE:      {rmse:.2f} m")
print(f"  MAE:       {mae:.2f} m")
print(f"\nİmar median height:  {comp['building_height_mean'].median():.2f} m")
print(f"GHSL median:         {comp['ghsl_mean'].median():.2f} m")

In [ ]:
# --- Görselleştirme: 4 panel ---
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
specs = [
    ("building_height_mean",     "İmar yüksekliği ortalama (m)", "viridis"),
    ("building_density_per_km2", "Yapı yoğunluğu (bina/km²)",     "OrRd"),
    ("ghsl_mean",                "GHSL yüksekliği (m)",          "viridis"),
    (None, None, None),  # son panel scatter olacak
]
for ax, (col, title, cmap) in zip(axes.ravel()[:3], specs[:3]):
    vals = grid[col].dropna()
    if len(vals) == 0:
        ax.set_title(f"{title} (boş)")
        continue
    grid.plot(
        column=col, cmap=cmap, ax=ax,
        legend=True, legend_kwds={"label": title, "shrink": 0.6},
        linewidth=0,
        vmin=np.percentile(vals, 2), vmax=np.percentile(vals, 98),
        missing_kwds={"color": "#eaeaea"},
    )
    boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color="#0a9396", linewidth=0.8)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.ticklabel_format(style="plain")

# 4. panel — imar vs GHSL scatter
ax = axes.ravel()[3]
sample = comp.sample(min(5000, len(comp)), random_state=42)
ax.scatter(sample["ghsl_mean"], sample["building_height_mean"], s=3, alpha=0.4, color="#264653")
lim = max(comp.max().max(), 1)
ax.plot([0, lim], [0, lim], "r--", linewidth=1, label="y = x")
ax.set_xlabel("GHSL height (m)")
ax.set_ylabel("İmar height_mean (m)")
ax.set_title(f"İmar vs GHSL  (r={r:.2f}, RMSE={rmse:.1f}m)")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES / "04_building_overview.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Kaydet + summary tablosu ---
grid.to_file(GRID_30M_BUILDING, driver="GPKG")
print(f"Kaydedildi: {GRID_30M_BUILDING} ({GRID_30M_BUILDING.stat().st_size/1024/1024:.2f} MB)")

summary = pd.DataFrame({
    "metrik": [
        "toplam_bina", "measured", "imputed",
        "cv_rmse_m", "cv_mae_m", "cv_r2",
        "grid_30m_hucre", "binali_hucre", "binali_hucre_pct",
        "medyan_yapi_yogunlugu_per_km2",
        "medyan_yukseklik_imar", "medyan_yukseklik_ghsl",
        "imar_vs_ghsl_pearson_r", "imar_vs_ghsl_rmse_m",
    ],
    "deger": [
        len(buildings),
        int((buildings["impute_source"]=="measured").sum()),
        int((buildings["impute_source"]=="imputed").sum()),
        round(cv["rmse"], 2), round(cv["mae"], 2), round(cv["r2"], 3),
        len(grid), int((grid["building_count"]>0).sum()),
        round(100*(grid["building_count"]>0).sum()/len(grid), 1),
        round(float(grid["building_density_per_km2"].median()), 1),
        round(float(comp["building_height_mean"].median()), 2),
        round(float(comp["ghsl_mean"].median()), 2),
        round(r, 3), round(rmse, 2),
    ],
})
summary.to_csv(TABLES / "04_building_summary.csv", index=False, encoding="utf-8-sig")
summary

## Özet

- 5,787 eksik kat kaydı **buffer cascade** (10/100/500/1000 m, min 3 komşu) ile dolduruldu.
- 5-fold CV ile imputation kalitesi raporlandı (RMSE/MAE/R²).
- GHSL JRC (2018) ile **bağımsız doğrulama**: imar-kaynaklı yükseklik vs GHSL global building height korelasyonu.
- 30 m grid'e 3 yeni kolon eklendi: `building_count`, `building_height_mean`, `building_density_per_km2`.

## Sınırlılıklar (tezde raporlanacak)

- TS 9111 dönüşümünde tüm binalar **konut** (3.0 m/kat) varsayıldı — veride kullanım amacı yok.
- Buffer cascade yerel yoğunluğa duyarlı; izole binalarda 1000 m fallback'i devreye girer.
- GHSL native 100 m, 30 m grid'e zonal mean ile downscale edildi → konum kayması olabilir.

## Sonraki Adım

`05_dtc_breeze.ipynb` — OSM coastline + ray casting ile rüzgar-yönelimli kıyı mesafesi (`DTC_breeze`); ayrıca OSM yol verisinden yol yoğunluğu hesabı.